### ライブラリ

In [ ]:
import pandas as pd

### データの読み込み

In [ ]:

df_bin_100 = pd.read_csv("../data/raw/train.csv", encoding='utf-8-sig')
df_bin_100.head()


,sample number,species number,樹種,含水率,4000-4100,4100-4200,4200-4300,4300-4400,4400-4500,4500-4600,...,9000-9100,9100-9200,9200-9300,9300-9400,9400-9500,9500-9600,9600-9700,9700-9800,9800-9900,9900-10000
0,1,1,イチョウ,216.129032,1.237228,1.195882,1.144523,1.080303,1.026959,1.000414,...,0.398909,0.397366,0.395788,0.395013,0.395481,0.398043,0.401804,0.405782,0.409845,0.413412
1,2,1,イチョウ,210.752688,1.207945,1.159597,1.111357,1.049931,0.999201,0.972858,...,0.404658,0.403267,0.401898,0.401181,0.401752,0.404235,0.407914,0.411909,0.415720,0.419084
2,3,1,イチョウ,205.913979,1.155000,1.105197,1.059435,1.002155,0.954435,0.929242,...,0.395227,0.393965,0.392542,0.392052,0.392489,0.394863,0.398281,0.401879,0.405611,0.408882
3,4,1,イチョウ,201.075269,1.104873,1.058732,1.014831,0.960605,0.915296,0.890062,...,0.386127,0.385018,0.383527,0.382973,0.383349,0.385650,0.389130,0.392596,0.395900,0.399082
4,5,1,イチョウ,196.236559,1.049295,0.994718,0.954323,0.906272,0.865555,0.840617,...,0.374985,0.373957,0.372548,0.371986,0.372400,0.374449,0.377664,0.381002,0.384058,0.386820


In [3]:
df_bin_100.shape

(1322, 64)

In [4]:

import numpy as np
import lightgbm as lgb

# データ読み込み
train = pd.read_csv("data/train.csv", encoding='cp932')
test  = pd.read_csv("data/test.csv",  encoding='cp932')

# 特徴量・目的変数の定義
meta_cols = ['sample number', 'species number', '樹種', '含水率']
feature_cols = [c for c in train.columns if c not in meta_cols]

X      = train[feature_cols].values
y      = train['含水率'].values
X_test = test[feature_cols].values

print(f"X shape:      {X.shape}")
print(f"y shape:      {y.shape}")
print(f"X_test shape: {X_test.shape}")


X shape:      (1322, 1555)
y shape:      (1322,)
X_test shape: (550, 1555)


In [7]:

import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

# trainの1割をvalidationとして分割
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
print(f"学習データ: {X_tr.shape[0]}件 / 評価データ: {X_val.shape[0]}件")

def objective(trial):
    params_optuna = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbose': -1,
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': 1,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
    }
    num_boost_round = trial.suggest_int('num_boost_round', 100, 2000)

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        params_optuna,
        dtrain,
        num_boost_round=num_boost_round,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest RMSE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


c:\Users\keisu\Desktop\Competition\スペクトル分析\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


学習データ: 1189件 / 評価データ: 133件


Best trial: 45. Best value: 8.56844: 100%|██████████| 50/50 [03:33<00:00,  4.27s/it]


Best RMSE: 8.5684
Best params: {'learning_rate': 0.021125820322112444, 'num_leaves': 35, 'min_child_samples': 16, 'feature_fraction': 0.910746860249163, 'bagging_fraction': 0.7424885459361689, 'lambda_l1': 0.00356130717551477, 'lambda_l2': 0.01165682085540909, 'num_boost_round': 804}


In [18]:

# ベストパラメータでtrain全体を再学習
best_params = study.best_params.copy()
num_boost_round_best = best_params.pop('num_boost_round')

best_params.update({
    'objective': 'regression',
    'metric': 'rmse',
    'verbose': -1,
    'random_state': 42,
    'bagging_freq': 1,
})

dtrain_full = lgb.Dataset(X, label=y)
model_best = lgb.train(
    best_params,
    dtrain_full,
    num_boost_round=num_boost_round_best,
)

# 予測
test_preds_best = model_best.predict(X_test)
print(f"予測件数: {len(test_preds_best)}")
print(f"予測値の範囲: {test_preds_best.min():.2f} 〜 {test_preds_best.max():.2f}")
print(f"予測値の平均: {test_preds_best.mean():.2f}")


予測件数: 550
予測値の範囲: 4.82 〜 171.09
予測値の平均: 51.84


In [19]:

import os

# 提出ファイルの作成
os.makedirs("data/submission", exist_ok=True)

submission = pd.DataFrame({
    'sample number': test['sample number'],
    '含水率': test_preds_best
})

output_path = "data/submission/submission1.csv"
submission.to_csv(output_path, index=False, header=False, encoding='utf-8-sig')
print(f"保存完了: {output_path}")
submission.head()


保存完了: data/submission/submission1.csv


,sample number,含水率
0,95,154.838514
1,96,150.803302
2,97,145.069758
3,98,153.225207
4,99,155.095235
